# Laboratorio Centinela — Fases 1, 2 y 3 en un solo cuaderno
**Profundización I: Deep Learning · Maestría en Ciencia de Datos · USTA**
*Tomás Rendón Bustamante* · *Juan Sebastián Gamba Aguirre*

Este cuaderno es una **herramienta para explorar y explicar** el proyecto durante la sustentación: se mueven deslizadores y se ve qué pasa. **No reemplaza** los cuadernos de las Fases 1–2 y 3 (ahí están los resultados oficiales). Cada sección dice qué se simplificó.

**Cómo usarlo (Google Colab):** 1) Menú *Entorno de ejecución → Ejecutar todo* (o ejecute las celdas en orden). 2) Acepte el permiso de Drive. 3) Mueva los deslizadores. Las secciones 1 y 2 no necesitan GPU; la 3 usa GPU si hay (*Entorno de ejecución → Cambiar tipo → T4*).

**Qué lee de su Drive** (carpeta `MyDrive/Centinela/`): la serie del IDEAM `0026177030-NIVEL_H.csv` (si no está, la celda 0.2 le deja subirla) y `Fase_3/centinela_recortes_sar_224.npz` (recortes de radar). No se necesitan claves de Copernicus ni de ningún servicio.

## 0. Preparación

In [ ]:
# 0.1 Librerías y carpeta del proyecto en Drive
import os, glob, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
warnings.filterwarnings('ignore')
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/Centinela'
except ImportError:                       # fuera de Colab: use la variable de entorno CENTINELA_BASE
    BASE = os.environ.get('CENTINELA_BASE', os.path.expanduser('~/Centinela'))
print('Carpeta del proyecto:', BASE, '| existe:', os.path.isdir(BASE))

In [ ]:
# 0.2 Localiza los archivos de datos (busca dentro de la carpeta del proyecto)
def buscar(nombre):
    hallados = glob.glob(os.path.join(BASE, '**', nombre), recursive=True)
    return hallados[0] if hallados else None

RUTA_IDEAM = buscar('0026177030-NIVEL_H.csv')
RUTA_NPZ = buscar('centinela_recortes_sar_224.npz')

if RUTA_IDEAM is None:
    try:                                   # en Colab: permite subir el CSV del IDEAM y lo guarda en Drive
        from google.colab import files
        print('No encontré el CSV del IDEAM. Seleccione 0026177030-NIVEL_H.csv')
        subido = files.upload()
        os.makedirs(BASE, exist_ok=True)
        RUTA_IDEAM = os.path.join(BASE, '0026177030-NIVEL_H.csv')
        with open(RUTA_IDEAM, 'wb') as f:
            f.write(list(subido.values())[0])
    except ImportError:
        pass

print('Serie del IDEAM  :', RUTA_IDEAM)
print('Recortes de radar:', RUTA_NPZ)
assert RUTA_IDEAM, 'Falta el CSV del IDEAM (0026177030-NIVEL_H.csv).'

## 1. Fase 1 — ¿Qué tan difícil es predecir el nivel del río?
Se usa la serie del IDEAM (solo registros **Definitivo**, 2020–2025), igual que en la Fase 1.

In [ ]:
# 1.1 Carga de la serie del IDEAM (solo definitivos 2020-2025)
raw = pd.read_csv(RUTA_IDEAM)
raw['fecha'] = pd.to_datetime(raw['fechaObservacion'])
d = raw[raw['nivelAprobacion'].astype(str).str.contains('Definitivo', case=False)]
d = d[(d['fecha'] >= '2020-01-01') & (d['fecha'] < '2026-01-01')].drop_duplicates('fecha')
serie = d.set_index('fecha')['valorObservado'].astype(float).sort_index()
print(f'Registros totales: {len(raw):,} | definitivos 2020-2025: {len(serie):,}'.replace(',', ' '))
print('Periodo:', serie.index.min(), '->', serie.index.max())

In [ ]:
# 1.2 Tabla de modelado: nivel actual, rezagos y objetivo a +h horas (unidos por fecha exacta)
REZAGOS = (12, 24, 48, 72)

def arma_tabla(h_horas):
    t = pd.Timedelta(hours=h_horas)
    df = pd.DataFrame({'nivel': serie})
    df['objetivo'] = serie.reindex(serie.index + t).values
    for r in REZAGOS:
        df[f'nivel_{r}h'] = serie.reindex(serie.index - pd.Timedelta(hours=r)).values
    df['fecha_objetivo'] = df.index + t
    df = df.dropna().sort_values('fecha_objetivo')
    df['cambio'] = df['objetivo'] - df['nivel']              # lo que queremos predecir (cm)
    df['cambio_previo'] = df['nivel'] - df['nivel_12h']      # cuánto subió en las 12 h anteriores
    return df

def particion(df):
    n = len(df); i1, i2 = int(0.70 * n), int(0.85 * n)       # cronológica 70/15/15
    return {'entrenamiento': df.iloc[:i1], 'validación': df.iloc[i1:i2], 'prueba': df.iloc[i2:], 'todo': df}

TABLAS = {h: arma_tabla(h) for h in (12, 24, 36)}
print({h: len(t) for h, t in TABLAS.items()})

In [ ]:
# 1.3 EXPLORADOR: ¿con qué frecuencia sube el río y qué tan buenas son las reglas simples?
def metricas(real, pred):
    vp = int(((real) & (pred)).sum()); fp = int((~real & pred).sum()); fn = int((real & ~pred).sum())
    prec = vp / (vp + fp) if vp + fp else 0.0; sens = vp / (vp + fn) if vp + fn else 0.0
    f1 = 2 * prec * sens / (prec + sens) if prec + sens else 0.0
    return vp, fp, fn, prec, sens, f1

def explorador(horizonte=12, umbral=4, conjunto='prueba'):
    df = particion(TABLAS[horizonte])[conjunto]
    real = (df['cambio'] > umbral).values
    print(f'{conjunto}: {len(df):,} ejemplos | el río sube más de {umbral} cm en {horizonte} h en el '
          f'{100 * real.mean():.1f} % de los casos'.replace(',', ' '))
    print(f'Error de la regla «el nivel seguirá igual» (persistencia): {df["cambio"].abs().mean():.2f} cm\n')
    reglas = {'«nunca sube»': np.zeros(len(df), bool),
              '«sube si ya subió > umbral en las 12 h previas»': (df['cambio_previo'] > umbral).values,
              '«siempre sube»': np.ones(len(df), bool)}
    filas = []
    for nombre, pred in reglas.items():
        vp, fp, fn, prec, sens, f1 = metricas(real, pred)
        filas.append([nombre, vp, fp, fn, round(prec, 3), round(sens, 3), round(f1, 3)])
    print(pd.DataFrame(filas, columns=['regla', 'VP', 'FP', 'FN', 'precisión', 'sensibilidad', 'F1']).to_string(index=False))
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
    ax[0].hist(df['cambio'].clip(-40, 40), bins=60, color='#8a8a8a')
    ax[0].axvline(umbral, color='#e07b00'); ax[0].set_xlabel(f'Cambio de nivel a +{horizonte} h (cm)'); ax[0].set_ylabel('casos')
    ax[0].set_title(f'Línea naranja = umbral de {umbral} cm')
    ax[1].plot(df['fecha_objetivo'], df['nivel'], color='#1f4e79', lw=0.7); ax[1].set_title('Nivel del río en el periodo elegido (cm)')
    plt.tight_layout(); plt.show()

w_h = widgets.Dropdown(options=[12, 24, 36], value=12, description='Horizonte (h)')
w_u = widgets.IntSlider(value=4, min=1, max=10, description='Umbral (cm)')
w_c = widgets.Dropdown(options=['entrenamiento', 'validación', 'prueba', 'todo'], value='prueba', description='Conjunto')
display(widgets.HBox([w_h, w_u, w_c]), widgets.interactive_output(explorador, {'horizonte': w_h, 'umbral': w_u, 'conjunto': w_c}))

**Para explicar:** mueva el umbral y el conjunto. En *prueba* sube más seguido que en *entrenamiento* (más crecidas): por eso los errores de prueba son mayores. Ninguna regla simple detecta bien los aumentos y avisa a tiempo, y eso es lo que el MLP de la Fase 1 tampoco logró superar de forma demostrable.

In [ ]:
# 1.4 MINI-MLP: sobreajuste y regularización (versión simplificada de la Fase 1: 6 predictores, sin Dropout)
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

COLS = ['nivel', 'nivel_12h', 'nivel_24h', 'nivel_48h', 'nivel_72h', 'cambio_previo']

def entrena_mlp(neuronas=16, l2=1e-4, epocas=150, parada_temprana=True, semilla=0):
    df = particion(TABLAS[12]); tr, va, te = df['entrenamiento'], df['validación'], df['prueba']
    esc = StandardScaler().fit(tr[COLS]); s_y = tr['cambio'].std()
    Xtr, Xva, Xte = (esc.transform(a[COLS]) for a in (tr, va, te))
    ytr, yva = tr['cambio'].values / s_y, va['cambio'].values / s_y
    red = MLPRegressor(hidden_layer_sizes=(neuronas,), alpha=l2, learning_rate_init=1e-3, random_state=semilla)
    curvas, mejor, mejor_ep, pesos, sin_mejora = [], np.inf, 0, None, 0
    for ep in range(1, epocas + 1):
        red.partial_fit(Xtr, ytr)
        e_tr = np.mean((red.predict(Xtr) - ytr) ** 2) * s_y ** 2
        e_va = np.mean((red.predict(Xva) - yva) ** 2) * s_y ** 2
        curvas.append((e_tr, e_va))
        if e_va < mejor:
            mejor, mejor_ep, sin_mejora = e_va, ep, 0
            pesos = ([c.copy() for c in red.coefs_], [b.copy() for b in red.intercepts_])
        else:
            sin_mejora += 1
        if parada_temprana and sin_mejora >= 15:
            break
    if parada_temprana and pesos is not None:                 # restaura la mejor época
        red.coefs_, red.intercepts_ = pesos
    pred = red.predict(Xte) * s_y
    return {'curvas': np.array(curvas), 'mejor_ep': mejor_ep, 'epocas': len(curvas),
            'mae_modelo': float(np.mean(np.abs(pred - te['cambio'].values))),
            'mae_persistencia': float(te['cambio'].abs().mean())}

def muestra_mlp(neuronas, l2, epocas, parada):
    r = entrena_mlp(neuronas, l2, epocas, parada)
    print(f'Épocas entrenadas: {r["epocas"]} | mejor época (validación): {r["mejor_ep"]}')
    print(f'Error en prueba: modelo {r["mae_modelo"]:.2f} cm  vs  persistencia {r["mae_persistencia"]:.2f} cm')
    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.plot(r['curvas'][:, 0], label='entrenamiento'); ax.plot(r['curvas'][:, 1], label='validación')
    ax.axvline(r['mejor_ep'] - 1, color='k', ls='--', lw=0.8, label='mejor época')
    ax.set_yscale('log'); ax.set_xlabel('época'); ax.set_ylabel('MSE (cm², log)'); ax.legend(frameon=False)
    plt.tight_layout(); plt.show()

w_n = widgets.IntSlider(value=16, min=2, max=128, description='Neuronas')
w_l = widgets.FloatLogSlider(value=1e-4, base=10, min=-6, max=0, step=0.25, description='L2 (alpha)')
w_e = widgets.IntSlider(value=150, min=10, max=400, step=10, description='Épocas')
w_p = widgets.Checkbox(value=True, description='Parada temprana')
boton = widgets.Button(description='Entrenar', button_style='primary')
salida_mlp = widgets.Output()
def al_pulsar(_):
    with salida_mlp:
        salida_mlp.clear_output(wait=True)
        muestra_mlp(w_n.value, w_l.value, w_e.value, w_p.value)
boton.on_click(al_pulsar)
display(widgets.VBox([widgets.HBox([w_n, w_l]), widgets.HBox([w_e, w_p, boton]), salida_mlp]))

**Para explicar:** sin parada temprana y con muchas épocas, la curva de validación se separa de la de entrenamiento: eso es **sobreajuste**. Subir el L2 o activar la parada temprana lo contiene. Compare el error de prueba con el de la persistencia: en la Fase 1 el MLP quedó cerca de ella y no la superó de forma demostrable. *Simplificación:* aquí hay 6 predictores y sin Dropout; el experimento oficial (con Dropout) está en el cuaderno de las Fases 1 y 2.

## 2. Fase 2 — El radar Sentinel-1
Se usan los recortes de 224 × 224 × 3 (VV, VH y VH−VV en dB) guardados en la Fase 3.

In [ ]:
# 2.1 Carga de los recortes de radar
assert RUTA_NPZ, 'Falta Fase_3/centinela_recortes_sar_224.npz en su Drive.'
datos = np.load(RUTA_NPZ, allow_pickle=True)
IMG = datos['X']; CAMBIO = datos['cambio_24h_cm'].astype(float)
ETIQ = datos['etiqueta'] if 'etiqueta' in datos.files else datos['y']
FECHAS = pd.to_datetime(datos['fecha'])
orden = np.argsort(FECHAS.values)                       # orden cronológico
print('Recortes:', IMG.shape, '| periodo:', FECHAS.min().date(), '->', FECHAS.max().date())
print('Etiquetas:', pd.Series(ETIQ).value_counts().to_dict())

In [ ]:
# 2.2 VISOR de escenas: qué «ve» el radar
CANALES = ['VV (dB)', 'VH (dB)', 'VH − VV (dB)']

def ver_escena(idx=0):
    i = orden[idx]
    fig, ax = plt.subplots(1, 3, figsize=(11, 3.8))
    for c in range(3):
        im = IMG[i, c]; lo, hi = np.nanpercentile(im, [2, 98])
        ax[c].imshow(im, cmap='gray', vmin=lo, vmax=hi); ax[c].set_title(CANALES[c]); ax[c].axis('off')
    fig.suptitle(f'Escena {idx + 1}/{len(orden)} · {FECHAS[i].date()} · etiqueta: {ETIQ[i]} · cambio de nivel a +24 h: {CAMBIO[i]:+.1f} cm')
    plt.tight_layout(); plt.show()

w_i = widgets.IntSlider(value=0, min=0, max=len(orden) - 1, description='Escena', layout=widgets.Layout(width='600px'))
display(w_i, widgets.interactive_output(ver_escena, {'idx': w_i}))

**Para explicar:** las imágenes se parecen mucho entre una fecha y otra y entre las clases (baja / estable / sube). Por eso el radar no se usa como foto, sino resumido; y por eso la rama de CNN sobre imágenes sola no encontró señal.

In [ ]:
# 2.3 «¿AYUDA EL RADAR?» — juguete didáctico: solo radar (6 resúmenes por escena) -> cambio a +24 h
from sklearn.linear_model import Ridge

RESUMEN = np.stack([IMG[:, c].reshape(len(IMG), -1).mean(1) for c in range(3)] +
                   [IMG[:, c].reshape(len(IMG), -1).std(1) for c in range(3)], axis=1)[orden]
Y = CAMBIO[orden]

def prueba_radar(alpha=1.0, barajar=False, semilla=0):
    n = len(Y); i1, i2 = int(0.70 * n), int(0.85 * n)           # cronológica 70/15/15
    X = RESUMEN.copy()
    if barajar:                                                    # control: mezcla las escenas (rompe la relación fecha-radar)
        X = X[np.random.default_rng(semilla).permutation(n)]
    mu, sd = X[:i1].mean(0), X[:i1].std(0) + 1e-9
    X = (X - mu) / sd
    m = Ridge(alpha=alpha).fit(X[:i1], Y[:i1])
    mae = lambda a, b: float(np.mean(np.abs(a - b)))
    return {'prueba_escenas': n - i2, 'mae_modelo': mae(m.predict(X[i2:]), Y[i2:]),
            'mae_cero': mae(np.zeros(n - i2), Y[i2:]), 'mae_mediana': mae(np.full(n - i2, np.median(Y[:i1])), Y[i2:])}

def muestra_radar(alpha, barajar, semilla):
    r = prueba_radar(alpha, barajar, semilla)
    print(f'Escenas de prueba: {r["prueba_escenas"]}')
    print(f'Error del modelo con radar{" BARAJADO" if barajar else ""}: {r["mae_modelo"]:.2f} cm')
    print(f'Referencias: «cambio = 0»: {r["mae_cero"]:.2f} cm | mediana de entrenamiento: {r["mae_mediana"]:.2f} cm')
    print('Con pocas escenas de prueba, diferencias de décimas de cm son ruido.')

w_a = widgets.FloatLogSlider(value=1.0, base=10, min=-2, max=4, step=0.25, description='Ridge alpha')
w_b = widgets.Checkbox(value=False, description='Barajar radar (control)')
w_s = widgets.IntSlider(value=0, min=0, max=20, description='Semilla')
display(widgets.HBox([w_a, w_b, w_s]), widgets.interactive_output(muestra_radar, {'alpha': w_a, 'barajar': w_b, 'semilla': w_s}))

**Para explicar:** mire si el modelo con *solo radar y resúmenes simples* supera a «cambio = 0»: en la Rama A de la Fase 2 las imágenes solas no lo lograron (sin evidencia). El hallazgo positivo de la Fase 2 viene de **combinar la serie del río con 20 valores de radar por distancia al cauce** y validar por bloques de tiempo (13,70 → 11,54 cm a +24 h; 21,02 → 17,78 cm a +36 h, 5 de 5 bloques). Esos resultados oficiales están en el cuaderno de las Fases 1 y 2; esta celda es solo una demostración y el control «barajar» muestra cómo se verifica que un resultado no sea casualidad.

## 3. Fase 3 — GPU, precisión mixta y modelos para el borde

In [ ]:
# 3.1 BANCO DE PRUEBAS EN VIVO: float32 frente a float16 (AMP) con ResNet-18 en esta misma GPU
import time
try:
    import torch, torchvision
    HAY_TORCH = True
except ImportError:
    HAY_TORCH = False
    print('PyTorch no está instalado en este entorno; use Google Colab.')

def bench(modo, lote, rep):
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    amp = (modo == 'float16 (AMP)') and dev == 'cuda'
    torch.manual_seed(0)
    red = torchvision.models.resnet18(weights=None, num_classes=3).to(dev).train()
    opt = torch.optim.Adam(red.parameters(), 1e-3); crit = torch.nn.CrossEntropyLoss()
    x = torch.randn(lote, 3, 224, 224, device=dev); y = torch.randint(0, 3, (lote,), device=dev)
    escala = torch.amp.GradScaler('cuda', enabled=amp) if dev == 'cuda' else None
    def paso():
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=dev, dtype=torch.float16, enabled=amp):
            perdida = crit(red(x), y)
        if escala is not None and amp:
            escala.scale(perdida).backward(); escala.step(opt); escala.update()
        else:
            perdida.backward(); opt.step()
    for _ in range(3): paso()                                    # calentamiento
    if dev == 'cuda':
        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    for _ in range(rep): paso()
    if dev == 'cuda': torch.cuda.synchronize()
    ms = (time.time() - t0) / rep * 1000
    vram = torch.cuda.max_memory_allocated() / 1e6 if dev == 'cuda' else float('nan')
    del red, opt, x, y
    if dev == 'cuda': torch.cuda.empty_cache()
    return ms, lote / ms * 1000, vram

def muestra_bench(lote, rep):
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('Dispositivo:', torch.cuda.get_device_name(0) if dev == 'cuda' else 'CPU (sin GPU: float16 no se usa)')
    filas = []
    for modo in ('float32', 'float16 (AMP)'):
        try:
            ms, ips, vram = bench(modo, lote, rep); filas.append([modo, round(ms, 1), round(ips), round(vram)])
        except RuntimeError as e:
            filas.append([modo, 'CUDA OOM' if 'out of memory' in str(e) else 'error', '', ''])
            if dev == 'cuda': torch.cuda.empty_cache()
    t = pd.DataFrame(filas, columns=['modo', 'ms por lote', 'imágenes/s', 'VRAM pico (MB)']); print(t.to_string(index=False))
    if len(filas) == 2 and isinstance(filas[0][1], float) and isinstance(filas[1][1], float):
        print(f'Aceleración float32 / float16: {filas[0][1] / filas[1][1]:.2f}×')

if HAY_TORCH:
    w_lote = widgets.Dropdown(options=[16, 32, 64, 128, 256, 512, 1024], value=32, description='Lote')
    w_rep = widgets.IntSlider(value=8, min=3, max=30, description='Repeticiones')
    b3 = widgets.Button(description='Medir', button_style='primary'); sal3 = widgets.Output()
    def al_medir(_):
        with sal3:
            sal3.clear_output(wait=True); muestra_bench(w_lote.value, w_rep.value)
    b3.on_click(al_medir)
    display(widgets.VBox([widgets.HBox([w_lote, w_rep, b3]), sal3]))

**Para explicar:** con lote 32 y 128 en la T4 el informe midió 1,97× y 2,09× al aislar el cómputo. Pruebe lote 1024: en float32 aparece **CUDA OOM** (se acaba la memoria) y en float16 todavía cabe. *Nota:* aquí se usan datos aleatorios y una ResNet-18 sin pesos preentrenados; mide velocidad y memoria, no calidad.

In [ ]:
# 3.2 Modelos exportados (lee los archivos reales de su Drive) y tabla antes/después de la compresión
carpeta_modelos = os.path.join(BASE, 'Fase_3', 'modelos')
archivos = sorted(glob.glob(os.path.join(carpeta_modelos, '*')))
print('Archivos en', carpeta_modelos)
for a in archivos:
    print(f'  {os.path.getsize(a) / 1e6:8.1f} MB  {os.path.basename(a)}')
if not archivos:
    print('  (no hay archivos: la tabla de abajo trae los valores medidos en la Fase 3)')

TABLA_F3 = pd.DataFrame([
    ['PyTorch float32 (.pth)', 44.8, 118.5, 0.293, 0.197, '0/11', '100 %'],
    ['ONNX float32 (.onnx)',   44.8,  35.3, 0.293, 0.197, '0/11', '100 %'],
    ['ONNX INT8 (.onnx)',      11.3,  23.8, 0.244, 0.131, '0/11', '95,1 %']],
    columns=['versión', 'tamaño (MB)', 'latencia mediana (ms)', 'exactitud prueba', 'F1 macro', 'aciertos «sube»', 'coincide con PyTorch'])
print('\nValores medidos en la Fase 3 (CPU de Colab, 1 hilo, lote 1; no es una tablet):')
print(TABLA_F3.to_string(index=False))

In [ ]:
# 3.3 ¿SIRVE COMO ALERTA? Un umbral de calidad frente al modelo de imagen de la Fase 3
def semaforo(exigida=0.60):
    modelo, referencia = 0.293, 0.439           # exactitud en prueba: modelo float32 y «siempre baja»
    print(f'Exactitud del modelo en prueba: {modelo:.3f} | referencia «siempre baja»: {referencia:.3f}')
    print(f'Exactitud mínima que exigimos: {exigida:.2f}')
    print('→ El modelo CUMPLE el mínimo.' if modelo >= exigida else '→ El modelo NO cumple el mínimo exigido.')
    print('→ Además no supera a la referencia trivial; acierta 0 de 11 imágenes «sube» (la clase que importa).')
    fig, ax = plt.subplots(figsize=(6, 1.6))
    ax.barh(['exigida', 'referencia', 'modelo'], [exigida, referencia, modelo], color=['#e07b00', '#8a8a8a', '#1f4e79'])
    ax.set_xlim(0, 1); ax.set_xlabel('exactitud'); plt.tight_layout(); plt.show()

w_x = widgets.FloatSlider(value=0.60, min=0.0, max=1.0, step=0.05, description='Exigida')
display(w_x, widgets.interactive_output(semaforo, {'exigida': w_x}))

**Para explicar:** la cadena técnica (GPU, precisión mixta, ONNX, INT8 de 44,8 a 11,3 MB y de 35,3 a 23,8 ms) funciona; el **modelo** no es apto como alerta (acierta 0 de 11 «sube»). La ruta de despliegue propuesta (imágenes sin conexión con LiteRT; serie por API en la nube) y la recomendación de no desplegar como alerta están en el informe de la Fase 3. LiteRT no se ejecutó.

---
**Resumen para la sustentación.** Fase 1: la historia del río no supera a «seguirá igual» y la formulación binaria no es adecuada. Fase 2: el radar ayuda a +24 y +36 h (−16 % y −15 % de error) con controles; las ramas A, B y C no dieron evidencia concluyente. Fase 3: el despliegue técnico funciona; el modelo de imagen no sirve como alerta. Todo es un ejercicio de clase con un cliente hipotético.